# SAID — Spectral-Attention Integrated Detector
## Kaggle P100 Training

**Enable GPU P100:** Settings → Accelerator → GPU P100

**Required dataset:** Add `mdhabibourrahman/object-detection-dataset` via sidebar → + Add Data

In [ ]:
# Cell 1 — Bootstrap: clone repo, verify datasets, write YAMLs, check modules
import subprocess, sys

# Clone repo
subprocess.run(['git', 'clone',
    'https://github.com/habibour/foggy_object-detection_yolox.git',
    '/kaggle/working/repo'], check=True)

# Run full setup (datasets, YAMLs, module sanity check)
subprocess.run(['python', '/kaggle/working/repo/kaggle_setup.py'], check=True)

In [ ]:
# Cell 2 — Stage 1: Pre-train on VOC-FOG  (~2.5 hrs on P100)
import subprocess
subprocess.run([
    'python', '/kaggle/working/train.py',
    '--stage',     'vocfog',
    '--s1-epochs', '50',
    '--batch',     '32',
    '--device',    '0',
    '--kaggle'
], check=True)

In [ ]:
# Cell 3 — Stage 2: Fine-tune on RTTS  (~2.5 hrs on P100)
# Val every 10ep | Test every 20ep | Save best every 5ep
import subprocess
subprocess.run([
    'python', '/kaggle/working/train.py',
    '--stage',     'rtts',
    '--epochs',    '100',
    '--batch',     '32',
    '--device',    '0',
    '--val-freq',  '10',
    '--test-freq', '20',
    '--save-freq', '5',
    '--kaggle'
], check=True)

In [ ]:
# Cell 4 — Final evaluation on RTTS val + test + VOC-FOG test
import subprocess
subprocess.run([
    'python', '/kaggle/working/train.py',
    '--stage',  'validate',
    '--batch',  '32',
    '--device', '0',
    '--kaggle'
], check=True)

In [ ]:
# Cell 5 — List output weights and eval log
import json
from pathlib import Path

work = Path('/kaggle/working')

print('=== Saved Weights ===')
for f in sorted(work.rglob('*.pt')):
    print(f'  {f.relative_to(work)}  ({f.stat().st_size/1e6:.1f} MB)')

print('\n=== Eval Log ===')
log = work / 'runs/said/stage2b_full/eval_log.csv'
if log.exists():
    print(log.read_text())
else:
    print('Not found yet')

print('\n=== Summary ===')
summary = work / 'runs/said/stage2b_full/said_summary.json'
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))
else:
    print('Not found yet')

In [ ]:
# Cell 6 — (Optional) Pull latest code changes and re-run from Stage 2
# Use this if you updated code on your Mac and pushed to GitHub
import subprocess, shutil
from pathlib import Path

subprocess.run(['git', '-C', '/kaggle/working/repo', 'pull'], check=True)
shutil.copytree('/kaggle/working/repo/said', '/kaggle/working/said', dirs_exist_ok=True)
shutil.copy2('/kaggle/working/repo/train.py', '/kaggle/working/train.py')

ver = subprocess.check_output(
    ['git', '-C', '/kaggle/working/repo', 'log', '--oneline', '-1']
).decode().strip()
print(f'Updated to: {ver}')